In [ ]:
## Evidence Contract Validation

Before vulnerability analysis begins, every source is wrapped in a
versioned Aegis Evidence Record. The contract captures authorization,
integrity, provenance, freshness, parser confidence, evidence conflicts,
and multidimensional trust.

AegisSec does not permit low-quality or unauthorized evidence to silently
flow into automated vulnerability decisions.

In [2]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.validation.evidence_schema_validator import EvidenceSchemaValidator

record_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "evidence"
    / "valid_requirements_evidence.json"
)

validator = EvidenceSchemaValidator(
    PROJECT_ROOT / "schemas" / "aegis_evidence_record.schema.json"
)

result = validator.validate_file(record_path)

print(json.dumps(result.to_dict(), indent=2))

{
  "schema_valid": true,
  "business_rules_valid": true,
  "status": "accepted",
  "errors": [],
  "warnings": []
}


In [ ]:
## Mission-Aware Asset Context Validation

A vulnerability cannot be prioritized responsibly using CVSS, EPSS, or
sector name alone.

AegisSec therefore validates an accountable Asset Context Record containing:

- mission function;
- exposure;
- operational criticality;
- public-service impact;
- patient-safety impact;
- mission-readiness impact;
- data sensitivity;
- resilience requirements;
- asset ownership;
- evidence provenance.

The sector label is explicitly prohibited from directly determining
priority. Only documented and evidence-supported consequences may affect
the final policy decision.

In [3]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.validation.asset_context_validator import AssetContextValidator

asset_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "assets"
    / "valid_healthcare_asset.json"
)

asset_validator = AssetContextValidator(
    PROJECT_ROOT
    / "schemas"
    / "aegis_asset_context.schema.json"
)

asset_result = asset_validator.validate_file(asset_path)

print(json.dumps(asset_result.to_dict(), indent=2))

{
  "schema_valid": true,
  "business_rules_valid": true,
  "status": "accepted_with_warnings",
  "errors": [],
  "warnings": [
    {
      "code": "SYNTHETIC_DEMO_CONTEXT",
      "message": "This asset is a controlled demonstration fixture and must not be presented as a live operational asset.",
      "field_path": "provenance.origin_type"
    }
  ]
}


In [ ]:
## Provenance-Aware Vulnerability Intelligence

AegisSec does not collapse CVSS, EPSS, KEV, exploitation evidence,
affected-version ranges, and remediation guidance into one unexplained score.

Each intelligence dimension retains:

- its original source;
- evidence reference;
- retrieval date;
- freshness;
- missing-data status;
- source authority;
- conflicts;
- transformation provenance.

Important decision rules:

- CVSS measures technical severity, not complete organizational risk.
- Missing EPSS remains unknown and is never converted to zero.
- Absence from CISA KEV is not proof that exploitation does not exist.
- KEV listing cannot coexist with a no-known-exploitation conclusion.
- High-severity source conflicts require quarantine or human review.

In [4]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.validation.vulnerability_intelligence_validator import (
    VulnerabilityIntelligenceValidator,
)

intelligence_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "intelligence"
    / "valid_log4shell_intelligence.json"
)

intelligence_validator = VulnerabilityIntelligenceValidator(
    PROJECT_ROOT
    / "schemas"
    / "aegis_vulnerability_intelligence.schema.json"
)

intelligence_result = intelligence_validator.validate_file(
    intelligence_path
)

print(json.dumps(intelligence_result.to_dict(), indent=2))

{
  "schema_valid": true,
  "business_rules_valid": true,
  "status": "accepted_with_warnings",
  "errors": [],
  "warnings": [
    {
      "code": "EPSS_MISSING",
      "message": "EPSS data is missing and must remain unknown. Human review or cautious policy handling is required.",
      "field_path": "epss.status"
    },
    {
      "code": "SYNTHETIC_INTELLIGENCE_FIXTURE",
      "message": "This vulnerability intelligence record is a controlled fixture and must not be presented as a live intelligence retrieval.",
      "field_path": "provenance.origin_type"
    }
  ]
}


In [ ]:
## Tamper-Evident Aegis Decision Record

Every AegisSec finding produces a versioned Decision Record that binds:

- validated source evidence;
- asset and mission context;
- vulnerability intelligence;
- component identity;
- affectedness;
- policy minimum action;
- optional ML advisory;
- decision arbitration;
- human disposition;
- cryptographic audit lineage.

The Decision Record prevents silent modification and preserves the exact
inputs, policy version, model status, uncertainty, review requirement and
final accountable disposition.

The machine-learning model is advisory only and cannot lower a
non-overridable policy floor.

In [5]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.domain.decision_hashing import verify_decision_record_hash
from src.validation.decision_record_validator import DecisionRecordValidator

decision_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "decisions"
    / "valid_log4shell_decision_record.json"
)

decision_validator = DecisionRecordValidator(
    schema_path=(
        PROJECT_ROOT
        / "schemas"
        / "aegis_decision_record.schema.json"
    ),
    project_root=PROJECT_ROOT,
)

decision_result = decision_validator.validate_file(
    decision_path
)

with decision_path.open("r", encoding="utf-8") as file:
    decision_record = json.load(file)

print("Decision ID:", decision_record["decision_id"])
print(
    "System priority:",
    decision_record["arbitration"]["final_system_priority"],
)
print(
    "System action:",
    decision_record["arbitration"]["final_action"],
)
print(
    "Human review required:",
    decision_record["arbitration"]["human_review_required"],
)
print(
    "Record hash verified:",
    verify_decision_record_hash(decision_record),
)
print()
print(json.dumps(decision_result.to_dict(), indent=2))

Decision ID: AEG-DEC-LOG4J-HOSP-001
System priority: CRITICAL
System action: ACT
Human review required: True
Record hash verified: True

{
  "schema_valid": true,
  "business_rules_valid": true,
  "status": "accepted_with_warnings",
  "errors": [],
  "warnings": [
    {
      "code": "INPUT_ACCEPTED_WITH_WARNINGS",
      "message": "Referenced asset_context contains 1 warning(s).",
      "field_path": "input_records.reference.0"
    },
    {
      "code": "INPUT_ACCEPTED_WITH_WARNINGS",
      "message": "Referenced vulnerability_intelligence contains 2 warning(s).",
      "field_path": "input_records.reference.1"
    },
    {
      "code": "INPUT_ACCEPTED_WITH_WARNINGS",
      "message": "Referenced evidence_record contains 1 warning(s).",
      "field_path": "input_records.reference.2"
    },
    {
      "code": "ML_ADVISORY_NOT_RUN",
      "message": "The ML advisory model has not been run. The Decision Record currently relies on policy.",
      "field_path": "ml_advisory.status"
   

In [6]:
## Strict Parsing and Missing-Value Governance

Raw vulnerability data frequently contains strings, blank cells,
Boolean-like values, malformed numbers, locale-dependent dates and
inconsistent missing-value markers.

AegisSec does not rely on implicit Python or pandas conversions.

Every raw field is classified as:

- **Parsed**: a valid typed value was produced;
- **Missing**: no value exists and no replacement was invented;
- **Invalid**: the supplied value is malformed or outside its permitted range.

Important invariants:

- `"False"` must parse as `False`, not Python truthiness `True`.
- Missing EPSS must remain `None`, not `0`.
- A real EPSS score of `0` remains a valid observed value.
- `NaN`, infinity and out-of-range values are rejected.
- Dates must use unambiguous ISO formats.
- Invalid rows do not enter policy or machine-learning pipelines.

SyntaxError: invalid syntax (2019326143.py, line 3)

In [7]:
import csv
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.vulnerability_feature_parser import parse_vulnerability_feature_row

parsing_fixture_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "parsing"
    / "strict_parsing_cases.csv"
)

parsing_demo_rows = []

with parsing_fixture_path.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    reader = csv.DictReader(file)

    for raw_row in reader:
        report = parse_vulnerability_feature_row(raw_row)

        parsing_demo_rows.append(
            {
                "finding_id": raw_row["finding_id"],
                "raw_kev_flag": raw_row["kev_flag"],
                "parsed_kev_flag": report.values["kev_flag"],
                "raw_epss": raw_row["epss_probability"],
                "parsed_epss": report.values["epss_probability"],
                "epss_status": (
                    report.outcomes["epss_probability"]
                    .status.value
                ),
                "accepted": report.accepted,
                "error_codes": ", ".join(
                    error.code
                    for error in report.errors
                ) or "None",
                "warning_count": len(report.warnings),
            }
        )

strict_parsing_demo_df = pd.DataFrame(parsing_demo_rows)

display(strict_parsing_demo_df)

,finding_id,raw_kev_flag,parsed_kev_flag,raw_epss,parsed_epss,epss_status,accepted,error_codes,warning_count
0,FND-STRICT-001,False,False,,NaN,missing,True,None,1
1,FND-STRICT-002,0,False,0,0.0,parsed,True,None,3
2,FND-STRICT-003,maybe,None,NaN,NaN,invalid,False,"UNRECOGNIZED_BOOLEAN_TOKEN, NON_FINITE_NUMERIC...",0


In [ ]:
## Multidimensional Evidence Trust

AegisSec does not use a single subjective trust label.

Evidence is evaluated across eight independently explainable dimensions:

1. authenticity;
2. integrity;
3. completeness;
4. freshness;
5. consistency;
6. source authority;
7. parser confidence;
8. component-identity confidence.

The overall score uses a weighted geometric mean, but the aggregate score
cannot override mandatory trust floors.

Examples:

- unauthorized evidence is rejected;
- failed integrity is rejected;
- an unresolved high-severity conflict is quarantined;
- stale evidence is quarantined;
- synthetic evidence is clearly disclosed;
- a low parser-confidence score cannot be hidden by high scores elsewhere.

This implements the principle that AegisSec must never produce
high-confidence automated recommendations from low-trust evidence.

In [8]:
import copy
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.governance.evidence_trust_engine import EvidenceTrustEngine

evidence_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "evidence"
    / "valid_requirements_evidence.json"
)

with evidence_path.open("r", encoding="utf-8") as file:
    base_evidence_record = json.load(file)

trust_engine = EvidenceTrustEngine(
    PROJECT_ROOT
    / "policies"
    / "trust"
    / "evidence_trust_policy_v1.yaml"
)

fixed_assessment_time = datetime(
    2026,
    7,
    13,
    18,
    45,
    tzinfo=timezone.utc,
)

high_trust_record = copy.deepcopy(base_evidence_record)
high_trust_record["evidence_id"] = "AEG-EVD-TRUST-HIGH-001"

stale_record = copy.deepcopy(base_evidence_record)
stale_record["evidence_id"] = "AEG-EVD-TRUST-STALE-001"
stale_record["freshness"]["status"] = "stale"
stale_record["freshness"]["age_seconds"] = 172800
stale_record["freshness"]["maximum_age_seconds"] = 86400

unauthorized_record = copy.deepcopy(base_evidence_record)
unauthorized_record["evidence_id"] = (
    "AEG-EVD-TRUST-UNAUTHORIZED-001"
)
unauthorized_record["authorization"]["status"] = "unauthorized"

trust_scenarios = {
    "High-trust controlled evidence": high_trust_record,
    "Stale evidence": stale_record,
    "Unauthorized evidence": unauthorized_record,
}

trust_summary_rows = []
trust_assessments = {}

for scenario_name, scenario_record in trust_scenarios.items():
    assessment = trust_engine.assess(
        scenario_record,
        assessed_at=fixed_assessment_time,
    )

    trust_assessments[scenario_name] = assessment

    trust_summary_rows.append(
        {
            "scenario": scenario_name,
            "aggregate_score": assessment.aggregate_score,
            "trust_level": assessment.trust_level,
            "action": assessment.action,
            "gates": ", ".join(
                gate.code
                for gate in assessment.gate_results
            ) or "None",
            "warnings": ", ".join(
                warning.code
                for warning in assessment.warnings
            ) or "None",
        }
    )

trust_summary_df = pd.DataFrame(trust_summary_rows)

display(trust_summary_df)

,scenario,aggregate_score,trust_level,action,gates,warnings
0,High-trust controlled evidence,0.9774,high,ACCEPT_WITH_WARNINGS,None,"SIGNATURE_NOT_VERIFIED, DECLARED_TRUST_MISMATCH"
1,Stale evidence,0.8276,low,QUARANTINE,"STALE_OR_EXPIRED_EVIDENCE, TRUST_DIMENSION_BEL...","SIGNATURE_NOT_VERIFIED, DECLARED_TRUST_MISMATCH"
2,Unauthorized evidence,0.8652,rejected,REJECT,"UNAUTHORIZED_EVIDENCE, TRUST_DIMENSION_BELOW_F...","SIGNATURE_NOT_VERIFIED, DECLARED_TRUST_MISMATCH"


In [9]:
high_trust_assessment = trust_assessments[
    "High-trust controlled evidence"
]

dimension_rows = []

for dimension_name, dimension in (
    high_trust_assessment.dimensions.items()
):
    dimension_rows.append(
        {
            "dimension": dimension_name,
            "score": dimension.score,
            "weight": dimension.weight,
            "mandatory_floor": dimension.minimum_score,
            "below_floor": dimension.below_floor,
            "reason_codes": ", ".join(
                dimension.reason_codes
            ),
        }
    )

trust_dimensions_df = pd.DataFrame(dimension_rows)

display(trust_dimensions_df)

,dimension,score,weight,mandatory_floor,below_floor,reason_codes
0,authenticity,0.9375,0.16,0.50,False,"AUTHORIZATION_AUTHORIZED, COLLECTION_BASIS_APP..."
1,integrity,1.0000,0.18,0.60,False,"INTEGRITY_VERIFIED, CRYPTOGRAPHIC_HASH_PRESENT..."
2,completeness,1.0000,0.12,0.50,False,"COMPONENT_VERSION_COVERAGE_EVALUATED, PURL_NOT..."
3,freshness,1.0000,0.12,0.50,False,"FRESHNESS_CURRENT, AGE_WITHIN_HALF_FRESHNESS_W..."
4,consistency,1.0000,0.14,0.50,False,NO_CONFLICTS_DETECTED
5,source_authority,0.9000,0.10,0.45,False,SOURCE_AUTHORITY_HIGH
6,parser_confidence,0.9800,0.10,0.60,False,PARSER_STATUS_SUCCESS
7,identity_confidence,1.0000,0.08,0.55,False,"VERSION_IDENTITY_COVERAGE_EVALUATED, NAME_VERS..."


In [ ]:
## Evidence-Governed Affectedness State Machine

AegisSec does not treat a package-name similarity or CVE lookup as proof
that an asset is affected.

The affectedness engine evaluates:

- trusted component identity;
- PURL or ecosystem/package matching;
- installed version;
- affected-version ranges;
- fixed-version boundaries;
- intelligence completeness;
- evidence-trust score;
- conflicting or unsupported ranges;
- runtime-presence evidence.

The engine may produce:

- Affected;
- Probably Affected;
- Unknown;
- Probably Not Affected;
- Not Affected;
- Fixed.

Safety invariants:

- No package match becomes Unknown, not Safe.
- Missing or malformed versions become Unknown.
- Conflicting or unsupported ranges trigger abstention.
- Quarantined or rejected evidence cannot support affectedness.
- Not Affected and Fixed require affirmative evidence.
- Probable and Unknown states require human review.

In [10]:
import copy
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.affectedness.affectedness_engine import AffectednessEngine
from src.governance.evidence_trust_engine import EvidenceTrustEngine

decision_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "decisions"
    / "valid_log4shell_decision_record.json"
)

intelligence_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "intelligence"
    / "valid_log4shell_intelligence.json"
)

evidence_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "evidence"
    / "valid_log4j_sbom_evidence.json"
)

with decision_path.open("r", encoding="utf-8") as file:
    decision_record = json.load(file)

with intelligence_path.open("r", encoding="utf-8") as file:
    intelligence_record = json.load(file)

with evidence_path.open("r", encoding="utf-8") as file:
    affectedness_evidence = json.load(file)

assessment_time = datetime(
    2026,
    7,
    13,
    19,
    0,
    tzinfo=timezone.utc,
)

evidence_trust = EvidenceTrustEngine().assess(
    affectedness_evidence,
    assessed_at=assessment_time,
)

affectedness_engine = AffectednessEngine()

base_component = decision_record["component_instance"]

scenario_components = {}

affected_component = copy.deepcopy(base_component)
affected_component["version"] = "2.14.1"
scenario_components["Affected version"] = (
    affected_component,
    intelligence_record,
)

fixed_component = copy.deepcopy(base_component)
fixed_component["version"] = "2.15.0"
scenario_components["Fixed version"] = (
    fixed_component,
    intelligence_record,
)

old_component = copy.deepcopy(base_component)
old_component["version"] = "1.2.17"
scenario_components["Before introduction"] = (
    old_component,
    intelligence_record,
)

unmatched_component = copy.deepcopy(base_component)
unmatched_component["name"] = "different-package"
unmatched_component["purl"] = (
    "pkg:maven/example/different-package@1.0.0"
)
scenario_components["No package match"] = (
    unmatched_component,
    intelligence_record,
)

partial_intelligence = copy.deepcopy(
    intelligence_record
)

partial_intelligence[
    "affected_packages"
][0]["range_status"] = "partial"

partial_component = copy.deepcopy(base_component)
partial_component["version"] = "2.14.1"

scenario_components["Partial source coverage"] = (
    partial_component,
    partial_intelligence,
)

affectedness_rows = []

for scenario_name, (
    scenario_component,
    scenario_intelligence,
) in scenario_components.items():
    assessment = affectedness_engine.assess(
        component_instance=scenario_component,
        intelligence_record=scenario_intelligence,
        evidence_trust=evidence_trust,
        assessed_at=assessment_time,
    )

    affectedness_rows.append(
        {
            "scenario": scenario_name,
            "version": scenario_component["version"],
            "status": assessment.status,
            "confidence": assessment.confidence,
            "package_match": (
                assessment.package_match.method
                if assessment.package_match
                else None
            ),
            "human_review_required": (
                assessment.human_review_required
            ),
            "reason_codes": ", ".join(
                assessment.reason_codes
            ),
        }
    )

affectedness_demo_df = pd.DataFrame(
    affectedness_rows
)

display(affectedness_demo_df)

,scenario,version,status,confidence,package_match,human_review_required,reason_codes
0,Affected version,2.14.1,affected,0.9794,exact_purl,False,"PACKAGE_IDENTITY_MATCHED, COMPONENT_VERSION_IN..."
1,Fixed version,2.15.0,fixed,0.9700,exact_purl,False,"VERSION_OUTSIDE_AFFECTED_RANGES, FIXED_BOUNDAR..."
2,Before introduction,1.2.17,not_affected,0.9500,exact_purl,False,"VERSION_PRECEDES_ALL_AFFECTED_RANGES, NOT_AFFE..."
3,No package match,2.14.1,unknown,NaN,None,True,"NO_AFFECTED_PACKAGE_MATCH, NO_MATCH_IS_NOT_PRO..."
4,Partial source coverage,2.14.1,probably_affected,0.7900,exact_purl,True,"PACKAGE_IDENTITY_MATCHED, COMPONENT_VERSION_IN..."


In [11]:
log4shell_assessment = affectedness_engine.assess(
    component_instance=affected_component,
    intelligence_record=intelligence_record,
    evidence_trust=evidence_trust,
    assessed_at=assessment_time,
)

range_reasoning_df = pd.DataFrame(
    [
        evaluation.to_dict()
        for evaluation
        in log4shell_assessment.range_evaluations
    ]
)

display(range_reasoning_df)

,range_index,range_type,introduced,fixed,last_affected,valid,affected,fixed_boundary_reached,before_introduced,reason_codes,error
0,0,ECOSYSTEM,2.0-beta9,2.3.1,None,True,False,True,False,[FIXED_BOUNDARY_REACHED],None
1,1,ECOSYSTEM,2.4,2.12.2,None,True,False,True,False,[FIXED_BOUNDARY_REACHED],None
2,2,ECOSYSTEM,2.13.0,2.15.0,None,True,True,False,False,[VERSION_WITHIN_AFFECTED_RANGE],None


In [ ]:
## Monotonic Policy Floor and Invariant Engine

AegisSec converts evidence and affectedness into an operational response
through versioned policy-as-code.

The policy engine evaluates:

- affectedness status;
- evidence trust;
- CISA KEV status;
- exploitation evidence;
- CVSS severity;
- EPSS availability;
- internet exposure;
- runtime reachability;
- asset criticality;
- mission essentiality;
- maximum operational or human consequence;
- remediation availability.

Policy rules establish minimum action, minimum priority, maximum response
time, review requirements and containment obligations.

Critical safeguards:

- rules may escalate but never lower an existing floor;
- KEV plus affectedness requires ACT and Critical-or-higher priority;
- unknown affectedness requires HOLD and human review;
- quarantined or rejected evidence blocks automation;
- missing EPSS remains unknown;
- sector labels are prohibited as direct policy inputs;
- catastrophic impact can establish Emergency response;
- all rule effects and evidence references are retained in the Decision Record.

In [12]:
import copy
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.policy.policy_floor_engine import PolicyFloorEngine

asset_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "assets"
    / "valid_healthcare_asset.json"
)

intelligence_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "intelligence"
    / "valid_log4shell_intelligence.json"
)

decision_path = (
    PROJECT_ROOT
    / "data"
    / "sample_inputs"
    / "decisions"
    / "valid_log4shell_decision_record_affectedness.json"
)

with asset_path.open("r", encoding="utf-8") as file:
    policy_asset = json.load(file)

with intelligence_path.open("r", encoding="utf-8") as file:
    policy_intelligence = json.load(file)

with decision_path.open("r", encoding="utf-8") as file:
    affectedness_decision = json.load(file)

policy_component = affectedness_decision[
    "component_instance"
]

policy_engine = PolicyFloorEngine()

policy_time = datetime(
    2026,
    7,
    13,
    19,
    15,
    tzinfo=timezone.utc,
)

accepted_trust = {
    "action": "ACCEPT",
    "aggregate_score": 0.95,
}

policy_scenarios = [
    {
        "name": "KEV affected",
        "asset": policy_asset,
        "intelligence": policy_intelligence,
        "affectedness": {
            "status": "affected",
            "confidence": 0.95,
            "supporting_evidence_ids": [
                "AEG-EVD-SBOM-LOG4J-001",
                "AEG-EVD-OSV-LOG4J-001",
            ],
        },
        "trust": accepted_trust,
    },
    {
        "name": "Unknown affectedness",
        "asset": policy_asset,
        "intelligence": policy_intelligence,
        "affectedness": {
            "status": "unknown",
            "confidence": None,
            "supporting_evidence_ids": [
                "AEG-EVD-SBOM-LOG4J-001",
            ],
        },
        "trust": accepted_trust,
    },
    {
        "name": "Fixed version",
        "asset": policy_asset,
        "intelligence": policy_intelligence,
        "affectedness": {
            "status": "fixed",
            "confidence": 0.95,
            "supporting_evidence_ids": [
                "AEG-EVD-SBOM-LOG4J-001",
                "AEG-EVD-OSV-LOG4J-001",
            ],
        },
        "trust": accepted_trust,
    },
    {
        "name": "Quarantined evidence",
        "asset": policy_asset,
        "intelligence": policy_intelligence,
        "affectedness": {
            "status": "affected",
            "confidence": 0.80,
            "supporting_evidence_ids": [
                "AEG-EVD-SBOM-LOG4J-001",
            ],
        },
        "trust": {
            "action": "QUARANTINE",
            "aggregate_score": 0.80,
        },
    },
]

policy_demo_rows = []

for scenario in policy_scenarios:
    assessment = policy_engine.evaluate(
        asset_context=scenario["asset"],
        intelligence_record=(
            scenario["intelligence"]
        ),
        component_instance=policy_component,
        affectedness_assessment=(
            scenario["affectedness"]
        ),
        evidence_trust=scenario["trust"],
        evaluated_at=policy_time,
    )

    policy_demo_rows.append(
        {
            "scenario": scenario["name"],
            "action": assessment.action,
            "minimum_priority": (
                assessment.minimum_priority
            ),
            "deadline_hours": (
                assessment.response_deadline_hours
            ),
            "human_review": (
                assessment.human_review_required
            ),
            "containment": (
                assessment.containment_required
            ),
            "closure_prohibited": (
                assessment.prohibit_closure
            ),
            "invariants_passed": (
                assessment.invariants_passed
            ),
            "rule_count": len(
                assessment.matched_rules
            ),
        }
    )

policy_demo_df = pd.DataFrame(
    policy_demo_rows
)

display(policy_demo_df)

,scenario,action,minimum_priority,deadline_hours,human_review,containment,closure_prohibited,invariants_passed,rule_count
0,KEV affected,ACT,CRITICAL,24.0,True,False,True,True,9
1,Unknown affectedness,HOLD,CRITICAL,24.0,True,False,True,True,2
2,Fixed version,TRACK,INFORMATIONAL,720.0,False,False,False,True,1
3,Quarantined evidence,HOLD,CRITICAL,4.0,True,False,True,True,10


In [13]:
log4shell_policy_assessment = (
    policy_engine.evaluate(
        asset_context=policy_asset,
        intelligence_record=(
            policy_intelligence
        ),
        component_instance=policy_component,
        affectedness_assessment={
            "status": "affected",
            "confidence": 0.95,
            "supporting_evidence_ids": [
                "AEG-EVD-SBOM-LOG4J-001",
                "AEG-EVD-OSV-LOG4J-001",
            ],
        },
        evidence_trust=accepted_trust,
        evaluated_at=policy_time,
    )
)

triggered_policy_rules_df = pd.DataFrame(
    [
        rule.to_dict()
        for rule
        in log4shell_policy_assessment.matched_rules
    ]
)

display(triggered_policy_rules_df)

,rule_id,rule_version,description,action_floor,priority_floor,deadline_hours,human_review_required,containment_required,prohibit_closure,non_overridable,supporting_evidence_ids
0,RULE-AFFECTED-BASELINE-001,1.0.0,Confirmed affectedness establishes an ATTEND a...,ATTEND,HIGH,168.0,False,False,False,True,"[AEG-EVD-SBOM-LOG4J-001, AEG-EVD-OSV-LOG4J-001]"
1,RULE-KEV-AFFECTED-001,1.0.0,A CISA KEV-listed vulnerability confirmed or p...,ACT,CRITICAL,24.0,True,False,True,True,"[AEG-EVD-KEV-LOG4J-001, AEG-EVD-SBOM-LOG4J-001..."
2,RULE-KNOWN-EXPLOITATION-001,1.0.0,Confirmed exploitation evidence requires ACT a...,ACT,CRITICAL,24.0,True,False,True,True,"[AEG-EVD-KEV-LOG4J-001, AEG-EVD-SBOM-LOG4J-001..."
3,RULE-INTERNET-EXPOSED-001,1.0.0,Internet-accessible affected components requir...,ACT,CRITICAL,24.0,True,False,True,True,"[AEG-EVD-REQ-000001, AEG-EVD-SBOM-LOG4J-001, A..."
4,RULE-SEVERE-IMPACT-001,1.0.0,"Severe mission, public-service, safety or oper...",ACT,CRITICAL,24.0,True,False,True,True,"[AEG-EVD-REQ-000001, AEG-EVD-SBOM-LOG4J-001, A..."
5,RULE-MISSION-ESSENTIAL-001,1.0.0,Mission-essential affected assets require at l...,ATTEND,HIGH,72.0,False,False,False,True,"[AEG-EVD-REQ-000001, AEG-EVD-SBOM-LOG4J-001, A..."
6,RULE-HIGH-CRITICALITY-001,1.0.0,High or very-high asset criticality requires a...,ATTEND,HIGH,72.0,False,False,False,True,"[AEG-EVD-REQ-000001, AEG-EVD-SBOM-LOG4J-001, A..."
7,RULE-CRITICAL-CVSS-001,1.0.0,Critical CVSS contributes an escalation floor ...,ATTEND,HIGH,72.0,False,False,False,False,"[AEG-EVD-NVD-LOG4J-001, AEG-EVD-SBOM-LOG4J-001..."
8,RULE-EPSS-MISSING-AFFECTED-001,1.0.0,Missing EPSS must remain unknown and requires ...,TRACK_STAR,MEDIUM,168.0,True,False,True,False,"[AEG-EVD-SBOM-LOG4J-001, AEG-EVD-OSV-LOG4J-001]"


In [14]:
policy_invariants_df = pd.DataFrame(
    [
        invariant.to_dict()
        for invariant
        in log4shell_policy_assessment.invariant_results
    ]
)

display(policy_invariants_df)

,code,passed,message
0,SECTOR_LABEL_EXCLUDED,True,Sector and subsector labels are excluded from ...
1,RULE_FLOORS_ENFORCED,True,Final action and priority meet or exceed every...
2,UNCERTAIN_AFFECTEDNESS_HELD,True,Unknown and probably-not-affected states requi...
3,KEV_AFFECTED_MINIMUM_ENFORCED,True,"KEV plus affectedness requires ACT-or-HOLD, Cr..."
4,LOW_TRUST_BLOCKS_AUTOMATION,True,Quarantined or rejected evidence blocks automa...
5,MISSING_EPSS_REMAINS_UNKNOWN,True,Missing EPSS remains None and is never interpr...
6,MONOTONIC_POLICY_STRUCTURE,True,Policy rules contain escalation floors only. N...


In [ ]:
## 120-Case Policy Assurance and Counterfactual Fairness Corpus

AegisSec does not evaluate policy safety using a few hand-selected examples.

The assurance corpus contains:

- five affectedness states;
- twelve operational risk profiles;
- two counterfactual sector labels;
- sixty matched counterfactual pairs;
- one hundred twenty total policy cases.

The operational profiles vary:

- CISA KEV listing;
- active exploitation;
- internet exposure;
- unknown exposure;
- catastrophic impact;
- severe impact;
- mission essentiality;
- asset criticality;
- critical CVSS;
- remediation unavailability;
- evidence quarantine or rejection;
- neutral baseline conditions.

For every pair, all decision-relevant inputs remain identical. Only the
sector label changes between healthcare and defence logistics.

A fairness failure is recorded if the sector change alters:

- action;
- priority;
- response deadline;
- review requirement;
- containment requirement;
- closure prohibition;
- triggered policy rules;
- invariant outcomes.

The expected decisions are generated by an independent declarative oracle,
not by copying the policy engine's result.

In [15]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assurance_report_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "assurance"
    / "step9_policy_assurance_report.json"
)

with assurance_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    assurance_report = json.load(file)

assurance_summary_df = pd.DataFrame(
    [
        {
            "metric": "Policy cases passed",
            "result": (
                f"{assurance_report['summary']['passed_cases']}"
                f"/{assurance_report['summary']['case_count']}"
            ),
        },
        {
            "metric": "Counterfactual pairs passed",
            "result": (
                f"{assurance_report['summary']['passed_pairs']}"
                f"/{assurance_report['summary']['pair_count']}"
            ),
        },
        {
            "metric": "Invariant cases passed",
            "result": (
                f"{assurance_report['summary']['invariant_pass_cases']}"
                f"/{assurance_report['summary']['case_count']}"
            ),
        },
        {
            "metric": "Sector disparities",
            "result": (
                assurance_report["summary"][
                    "sector_disparity_count"
                ]
            ),
        },
        {
            "metric": "Manifest checks passed",
            "result": (
                f"{assurance_report['summary']['manifest_checks_passed']}"
                f"/{assurance_report['summary']['manifest_check_count']}"
            ),
        },
        {
            "metric": "Overall assurance result",
            "result": (
                assurance_report["summary"][
                    "overall_passed"
                ]
            ),
        },
    ]
)

display(assurance_summary_df)

,metric,result
0,Policy cases passed,120/120
1,Counterfactual pairs passed,60/60
2,Invariant cases passed,120/120
3,Sector disparities,0
4,Manifest checks passed,7/7
5,Overall assurance result,True


In [16]:
affectedness_coverage_df = pd.DataFrame(
    [
        {
            "affectedness_profile": name,
            "case_count": count,
        }
        for name, count
        in assurance_report["coverage"][
            "affectedness_profiles"
        ].items()
    ]
)

operational_coverage_df = pd.DataFrame(
    [
        {
            "operational_profile": name,
            "case_count": count,
        }
        for name, count
        in assurance_report["coverage"][
            "operational_profiles"
        ].items()
    ]
)

display(affectedness_coverage_df)
display(operational_coverage_df)

,affectedness_profile,case_count
0,affected,24
1,fixed,24
2,probably_affected,24
3,probably_not_affected,24
4,unknown,24


,operational_profile,case_count
0,active_exploitation,10
1,baseline,10
2,catastrophic_impact,10
3,critical_cvss,10
4,exposure_unknown,10
5,high_criticality,10
6,internet_exposed,10
7,kev_listed,10
8,mission_essential,10
9,no_fix_available,10


In [17]:
case_results_df = pd.DataFrame(
    [
        {
            "affectedness": result[
                "affectedness_profile"
            ],
            "operational_profile": result[
                "operational_profile"
            ],
            "sector": result[
                "sector_label"
            ],
            "action": result[
                "actual"
            ]["action"],
            "priority": result[
                "actual"
            ]["priority"],
            "deadline_hours": result[
                "actual"
            ]["deadline_hours"],
            "human_review": result[
                "actual"
            ]["human_review_required"],
            "passed": result["passed"],
        }
        for result
        in assurance_report["case_results"]
    ]
)

healthcare_matrix_df = (
    case_results_df[
        case_results_df["sector"]
        == "healthcare"
    ]
    .pivot(
        index="affectedness",
        columns="operational_profile",
        values="action",
    )
)

display(healthcare_matrix_df)

operational_profile,active_exploitation,baseline,catastrophic_impact,critical_cvss,exposure_unknown,high_criticality,internet_exposed,kev_listed,mission_essential,no_fix_available,severe_impact,trust_blocked
affectedness,,,,,,,,,,,,
affected,ACT,ATTEND,ACT,ATTEND,ATTEND,ATTEND,ACT,ACT,ATTEND,ACT,ACT,HOLD
fixed,TRACK,TRACK,TRACK,TRACK,TRACK,TRACK,TRACK,TRACK,TRACK,TRACK,TRACK,HOLD
probably_affected,ACT,ATTEND,ACT,ATTEND,ATTEND,ATTEND,ACT,ACT,ATTEND,ACT,ACT,HOLD
probably_not_affected,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD
unknown,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD,HOLD


In [18]:
counterfactual_pairs_df = pd.DataFrame(
    assurance_report[
        "counterfactual_results"
    ]
)

display(
    counterfactual_pairs_df[
        [
            "pair_id",
            "sectors",
            "decision_equal",
            "matched_rules_equal",
            "counterfactual_key_equal",
            "passed",
        ]
    ]
)

,pair_id,sectors,decision_equal,matched_rules_equal,counterfactual_key_equal,passed
0,PAIR-001,"[defence_logistics, healthcare]",True,True,True,True
1,PAIR-002,"[defence_logistics, healthcare]",True,True,True,True
2,PAIR-003,"[defence_logistics, healthcare]",True,True,True,True
3,PAIR-004,"[defence_logistics, healthcare]",True,True,True,True
4,PAIR-005,"[defence_logistics, healthcare]",True,True,True,True
5,PAIR-006,"[defence_logistics, healthcare]",True,True,True,True
6,PAIR-007,"[defence_logistics, healthcare]",True,True,True,True
7,PAIR-008,"[defence_logistics, healthcare]",True,True,True,True
8,PAIR-009,"[defence_logistics, healthcare]",True,True,True,True
9,PAIR-010,"[defence_logistics, healthcare]",True,True,True,True


In [ ]:
## Fairness, Safety, Robustness and Release Governance

AegisSec converts the 120-case assurance corpus into release-blocking
evaluation metrics.

The evaluator measures:

- exact policy-output accuracy;
- action, priority and deadline agreement;
- under-triage and over-triage;
- safety recall;
- HOLD precision and recall;
- counterfactual sector consistency;
- deadline, review, containment and closure parity;
- escalation monotonicity;
- subgroup performance;
- required and total policy-rule coverage;
- pair-preserving bootstrap confidence intervals;
- mandatory quality-gate outcomes.

The controlled Step 10 assurance gate may pass while production readiness
remains blocked.

This distinction prevents a laboratory result from being represented as
government authorization or real-world operational validation.

Current fairness scope:

- the evaluation tests whether irrelevant sector labels alter deterministic
  policy decisions;
- it does not establish demographic fairness;
- it does not yet evaluate an ML model;
- it does not replace independent security assessment or operational testing.

In [19]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

step10_report_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "fairness"
    / "step10_fairness_robustness_report.json"
)

with step10_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    step10_report = json.load(file)

step10_summary_df = pd.DataFrame(
    [
        {
            "metric": "Exact policy cases",
            "result": (
                f"{step10_report['exact_match_metrics']['exact_case_count']}"
                f"/{step10_report['exact_match_metrics']['case_count']}"
            ),
        },
        {
            "metric": "Under-triage cases",
            "result": (
                step10_report[
                    "safety_metrics"
                ]["under_triage_count"]
            ),
        },
        {
            "metric": "Safety recall",
            "result": (
                step10_report[
                    "safety_metrics"
                ]["safety_recall"]
            ),
        },
        {
            "metric": "Counterfactual consistency",
            "result": (
                step10_report[
                    "counterfactual_metrics"
                ][
                    "counterfactual_consistency_rate"
                ]
            ),
        },
        {
            "metric": "Sector disparities",
            "result": (
                step10_report[
                    "counterfactual_metrics"
                ]["sector_disparity_count"]
            ),
        },
        {
            "metric": "Required rule coverage",
            "result": (
                f"{step10_report['rule_coverage']['covered_required_rule_count']}"
                f"/{step10_report['rule_coverage']['required_rule_count']}"
            ),
        },
        {
            "metric": "Full policy-rule coverage",
            "result": (
                f"{step10_report['rule_coverage']['covered_policy_rule_count']}"
                f"/{step10_report['rule_coverage']['policy_rule_count']}"
            ),
        },
        {
            "metric": "Stage assurance gate",
            "result": (
                step10_report[
                    "release_decision"
                ]["stage_gate_status"]
            ),
        },
        {
            "metric": "Production readiness",
            "result": (
                step10_report[
                    "release_decision"
                ][
                    "production_readiness_status"
                ]
            ),
        },
    ]
)

display(step10_summary_df)

,metric,result
0,Exact policy cases,120/120
1,Under-triage cases,0
2,Safety recall,1.0
3,Counterfactual consistency,1.0
4,Sector disparities,0
5,Required rule coverage,17/17
6,Full policy-rule coverage,17/22
7,Stage assurance gate,PASS
8,Production readiness,BLOCKED


In [20]:
quality_gates_df = pd.DataFrame(
    step10_report[
        "quality_gates"
    ]
)

display(
    quality_gates_df[
        [
            "gate_id",
            "metric",
            "operator",
            "threshold",
            "observed",
            "passed",
        ]
    ]
)

,gate_id,metric,operator,threshold,observed,passed
0,GATE-CASE-EXACT-001,case_exact_match_rate,gte,1.00,1.0,True
1,GATE-ACTION-EXACT-001,action_exact_match_rate,gte,1.00,1.0,True
2,GATE-PRIORITY-EXACT-001,priority_exact_match_rate,gte,1.00,1.0,True
3,GATE-DEADLINE-EXACT-001,deadline_exact_match_rate,gte,1.00,1.0,True
4,GATE-INVARIANTS-001,invariant_pass_rate,gte,1.00,1.0,True
5,GATE-UNDER-TRIAGE-001,under_triage_count,lte,0.00,0.0,True
6,GATE-SAFETY-RECALL-001,safety_recall,gte,1.00,1.0,True
7,GATE-COUNTERFACTUAL-001,counterfactual_consistency_rate,gte,1.00,1.0,True
8,GATE-SECTOR-DISPARITY-001,sector_disparity_count,lte,0.00,0.0,True
9,GATE-DEADLINE-PARITY-001,deadline_pair_max_gap_hours,lte,0.00,0.0,True


In [21]:
sector_subgroups_df = pd.DataFrame(
    [
        {
            "sector": sector,
            **metrics,
        }
        for sector, metrics
        in step10_report[
            "subgroup_metrics"
        ]["by_sector"].items()
    ]
)

display(
    sector_subgroups_df[
        [
            "sector",
            "case_count",
            "exact_match_rate",
            "under_triage_count",
            "under_triage_rate",
            "mean_deadline_hours",
        ]
    ]
)

,sector,case_count,exact_match_rate,under_triage_count,under_triage_rate,mean_deadline_hours
0,defence_logistics,60,1.0,0,0.0,166.2
1,healthcare,60,1.0,0,0.0,166.2


In [22]:
bootstrap_rows = []

for metric_name in (
    "case_exact_match_rate",
    "counterfactual_consistency_rate",
    "safety_recall",
):
    values = step10_report[
        "bootstrap"
    ][metric_name]

    bootstrap_rows.append(
        {
            "metric": metric_name,
            "mean": values["mean"],
            "lower_bound": (
                values["lower_bound"]
            ),
            "upper_bound": (
                values["upper_bound"]
            ),
        }
    )

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

display(bootstrap_df)

,metric,mean,lower_bound,upper_bound
0,case_exact_match_rate,1.0,1.0,1.0
1,counterfactual_consistency_rate,1.0,1.0,1.0
2,safety_recall,1.0,1.0,1.0


In [23]:
production_blockers_df = pd.DataFrame(
    {
        "production_blocking_reason": (
            step10_report[
                "release_decision"
            ]["blocking_reasons"]
        )
    }
)

display(production_blockers_df)

,production_blocking_reason
0,Full decision-policy rule coverage has not yet...
1,Real operational validation has not yet been c...
2,Independent security review has not yet been c...


## Step 11A: Complete Policy-Rule Coverage

Step 11A extends the controlled assurance corpus with targeted, counterfactually paired scenarios for every decision-policy rule that was not exercised during Step 9.

The evaluation proves full rule coverage without removing the remaining production blockers for real operational validation and independent security review.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

step11a_manifest_path = (
    PROJECT_ROOT
    / "data"
    / "assurance"
    / "step11a_policy_rule_coverage_manifest.json"
)

step11a_assurance_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "assurance"
    / "step11a_policy_rule_coverage_report.json"
)

step11a_evaluation_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "fairness"
    / "step11a_rule_coverage_evaluation_report.json"
)

step11a_manifest = json.loads(
    step11a_manifest_path.read_text(encoding="utf-8")
)

step11a_assurance = json.loads(
    step11a_assurance_path.read_text(encoding="utf-8")
)

step11a_evaluation = json.loads(
    step11a_evaluation_path.read_text(encoding="utf-8")
)

assurance_summary = step11a_assurance["summary"]
rule_coverage = step11a_evaluation["rule_coverage"]
safety_metrics = step11a_evaluation["safety_metrics"]
counterfactual_metrics = step11a_evaluation[
    "counterfactual_metrics"
]
release_decision = step11a_evaluation["release_decision"]

step11a_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Policy cases passed",
            "Observed": (
                f"{assurance_summary['passed_cases']}/"
                f"{assurance_summary['case_count']}"
            ),
        },
        {
            "Metric": "Counterfactual pairs passed",
            "Observed": (
                f"{assurance_summary['passed_pairs']}/"
                f"{assurance_summary['pair_count']}"
            ),
        },
        {
            "Metric": "Policy invariants passed",
            "Observed": (
                f"{assurance_summary['invariant_pass_cases']}/"
                f"{assurance_summary['case_count']}"
            ),
        },
        {
            "Metric": "Policy rules covered",
            "Observed": (
                f"{rule_coverage['covered_policy_rule_count']}/"
                f"{rule_coverage['policy_rule_count']}"
            ),
        },
        {
            "Metric": "Under-triage cases",
            "Observed": safety_metrics["under_triage_count"],
        },
        {
            "Metric": "Safety recall",
            "Observed": safety_metrics["safety_recall"],
        },
        {
            "Metric": "Counterfactual consistency",
            "Observed": counterfactual_metrics[
                "counterfactual_consistency_rate"
            ],
        },
        {
            "Metric": "Sector disparities",
            "Observed": counterfactual_metrics[
                "sector_disparity_count"
            ],
        },
        {
            "Metric": "Controlled stage gate",
            "Observed": release_decision["stage_gate_status"],
        },
        {
            "Metric": "Production readiness",
            "Observed": release_decision[
                "production_readiness_status"
            ],
        },
    ]
)

display(step11a_summary_df)

In [ ]:
targeted_profiles_df = pd.DataFrame(
    step11a_manifest["design"]["targeted_profiles"]
).rename(
    columns={
        "operational_profile": "Operational profile",
        "affectedness_profile": "Affectedness profile",
        "target_rule_id": "Target policy rule",
    }
)

display(targeted_profiles_df)

print(
    "Targeted rule coverage complete:",
    step11a_assurance["targeted_rule_coverage"]["complete"],
)

print(
    "Uncovered policy rules:",
    rule_coverage["uncovered_policy_rules"],
)

In [ ]:
production_blockers_df = pd.DataFrame(
    {
        "Remaining production blocker": (
            release_decision["blocking_reasons"]
        )
    }
)

display(production_blockers_df)

assert rule_coverage["full_rule_coverage_rate"] == 1.0
assert safety_metrics["under_triage_count"] == 0
assert safety_metrics["safety_recall"] == 1.0
assert counterfactual_metrics["sector_disparity_count"] == 0
assert release_decision["stage_gate_status"] == "PASS"
assert (
    release_decision["production_readiness_status"]
    == "BLOCKED"
)

print("Step 11A notebook assurance checks: PASS")

## Step 11B: Critical Policy Mutation Testing

This stage deliberately weakens critical decision-policy controls and verifies that the assurance system blocks or detects every dangerous mutation.

Mutations are executed only against temporary policy copies. The trusted policy remains unchanged.

In [25]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

step11b_report_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "mutation"
    / "step11b_policy_mutation_report.json"
)

step11b_results_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "mutation"
    / "step11b_mutation_results.csv"
)

step11b_report = json.loads(
    step11b_report_path.read_text(encoding="utf-8")
)

step11b_results_df = pd.read_csv(step11b_results_path)
step11b_summary = step11b_report["summary"]
step11b_release = step11b_report["release_decision"]

step11b_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Critical mutations",
            "Observed": step11b_summary[
                "critical_mutation_count"
            ],
        },
        {
            "Metric": "Killed by assurance",
            "Observed": step11b_summary[
                "killed_by_assurance_count"
            ],
        },
        {
            "Metric": "Blocked at policy load",
            "Observed": step11b_summary[
                "blocked_at_load_count"
            ],
        },
        {
            "Metric": "Surviving mutations",
            "Observed": step11b_summary[
                "survived_count"
            ],
        },
        {
            "Metric": "Harness errors",
            "Observed": step11b_summary[
                "harness_error_count"
            ],
        },
        {
            "Metric": "Critical defence rate",
            "Observed": step11b_summary[
                "critical_mutation_defence_rate"
            ],
        },
        {
            "Metric": "Executable kill rate",
            "Observed": step11b_summary[
                "executable_critical_mutation_kill_rate"
            ],
        },
        {
            "Metric": "Trusted policy integrity",
            "Observed": step11b_summary[
                "trusted_policy_integrity_passed"
            ],
        },
    ]
)

display(step11b_summary_df)

,Metric,Observed
0,Critical mutations,17
1,Killed by assurance,16
2,Blocked at policy load,1
3,Surviving mutations,0
4,Harness errors,0
5,Critical defence rate,1.0
6,Executable kill rate,1.0
7,Trusted policy integrity,True


In [26]:
preferred_columns = [
    "mutation_id",
    "mutation_family",
    "severity",
    "outcome",
    "detection_oracles",
]

available_columns = [
    column
    for column in preferred_columns
    if column in step11b_results_df.columns
]

display(step11b_results_df[available_columns])

outcome_counts_df = (
    step11b_results_df["outcome"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Count")
)

display(outcome_counts_df)

,mutation_id,severity,detection_oracles
0,MUT-KEV-ACTION-DOWNGRADE-001,critical,CASE_ORACLE|POLICY_INVARIANT|RELEASE_GATE|STAT...
1,MUT-KEV-PRIORITY-DOWNGRADE-001,critical,CASE_ORACLE|POLICY_INVARIANT|RELEASE_GATE|STAT...
2,MUT-ACTIVE-DEADLINE-EXTENSION-001,critical,CASE_ORACLE|RELEASE_GATE|STATIC_SECURITY_CONTR...
3,MUT-ACTIVE-REVIEW-REMOVAL-001,critical,CASE_ORACLE|RELEASE_GATE|STATIC_SECURITY_CONTR...
4,MUT-ACTIVE-CONTAINMENT-REMOVAL-001,critical,CASE_ORACLE|RELEASE_GATE|STATIC_SECURITY_CONTR...
5,MUT-ACTIVE-CLOSURE-WEAKENING-001,critical,CASE_ORACLE|RELEASE_GATE|STATIC_SECURITY_CONTR...
6,MUT-UNKNOWN-AFFECTEDNESS-TRACK-001,critical,CASE_ORACLE|POLICY_INVARIANT|RELEASE_GATE|STAT...
7,MUT-REJECTED-EVIDENCE-CONTINUE-001,critical,CASE_ORACLE|POLICY_INVARIANT|RELEASE_GATE|UNDE...
8,MUT-EPSS-MISSING-AS-ZERO-001,critical,CASE_ORACLE|RELEASE_GATE|RULE_COVERAGE|UNDER_T...
9,MUT-NO-FIX-CONTAINMENT-REMOVAL-001,critical,CASE_ORACLE|RELEASE_GATE|STATIC_SECURITY_CONTR...


KeyError: 'outcome'

In [27]:
remaining_blockers_df = pd.DataFrame(
    {
        "Remaining production blocker": (
            step11b_release["blocking_reasons"]
        )
    }
)

display(remaining_blockers_df)

assert step11b_summary["critical_mutation_count"] == 17
assert step11b_summary["killed_by_assurance_count"] == 16
assert step11b_summary["blocked_at_load_count"] == 1
assert step11b_summary["survived_count"] == 0
assert step11b_summary["harness_error_count"] == 0
assert (
    step11b_summary["critical_mutation_defence_rate"]
    == 1.0
)
assert (
    step11b_summary[
        "executable_critical_mutation_kill_rate"
    ]
    == 1.0
)
assert step11b_summary[
    "trusted_policy_integrity_passed"
] is True
assert step11b_summary["overall_passed"] is True
assert step11b_release["stage_gate_status"] == "PASS"
assert (
    step11b_release["production_readiness_status"]
    == "BLOCKED"
)

print("Step 11B mutation assurance checks: PASS")

,Remaining production blocker
0,Real operational validation has not yet been c...
1,Independent security review has not yet been c...


Step 11B mutation assurance checks: PASS


## Step 11F: Compound Attack-Chain Assurance

This stage combines policy mutation, evidence tampering, hostile-input manipulation and metamorphic transformations into ordered multi-layer attack chains.

Each chain is executed in forward and reverse order. The system must remain fail-closed, activate multiple independent controls and prevent every unsafe decision from becoming releasable.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

step11f_report_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "compound_attacks"
    / "step11f_compound_attack_report.json"
)

step11f_results_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "compound_attacks"
    / "step11f_chain_results.csv"
)

step11f_report = json.loads(
    step11f_report_path.read_text(encoding="utf-8")
)

step11f_results_df = pd.read_csv(step11f_results_path)
step11f_summary = step11f_report["summary"]
step11f_release = step11f_report["release_decision"]

step11f_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Critical chains",
            "Observed": step11f_summary[
                "critical_chain_count"
            ],
        },
        {
            "Metric": "Defended",
            "Observed": step11f_summary[
                "defended_critical_count"
            ],
        },
        {
            "Metric": "Blocked before decision",
            "Observed": step11f_summary[
                "blocked_before_decision_count"
            ],
        },
        {
            "Metric": "Rejected by assurance",
            "Observed": step11f_summary[
                "rejected_by_assurance_count"
            ],
        },
        {
            "Metric": "Quarantined",
            "Observed": step11f_summary[
                "quarantined_count"
            ],
        },
        {
            "Metric": "Undetected chains",
            "Observed": step11f_summary[
                "undetected_count"
            ],
        },
        {
            "Metric": "Harness errors",
            "Observed": step11f_summary[
                "harness_error_count"
            ],
        },
        {
            "Metric": "Unsafe decision releases",
            "Observed": step11f_summary[
                "unsafe_decision_release_count"
            ],
        },
        {
            "Metric": "Compound-chain defence rate",
            "Observed": step11f_summary[
                "critical_compound_chain_defence_rate"
            ],
        },
        {
            "Metric": "Fail-closed handling rate",
            "Observed": step11f_summary[
                "fail_closed_chain_handling_rate"
            ],
        },
        {
            "Metric": "Multi-control detection rate",
            "Observed": step11f_summary[
                "multi_control_detection_rate"
            ],
        },
        {
            "Metric": "Order consistency",
            "Observed": step11f_summary[
                "order_sensitive_chain_consistency_rate"
            ],
        },
        {
            "Metric": "Cross-layer integrity defence",
            "Observed": step11f_summary[
                "cross_layer_integrity_defence_rate"
            ],
        },
    ]
)

display(step11f_summary_df)


In [ ]:
preferred_columns = [
    "chain_id",
    "family",
    "severity",
    "outcome",
    "first_blocking_control",
    "activated_layers",
    "detection_oracles",
    "order_consistent",
    "unsafe_decision_released",
]

available_columns = [
    column
    for column in preferred_columns
    if column in step11f_results_df.columns
]

display(step11f_results_df[available_columns])

outcome_counts_df = (
    step11f_results_df["outcome"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Count")
)

display(outcome_counts_df)


In [ ]:
remaining_blockers_df = pd.DataFrame(
    {
        "Remaining production blocker": (
            step11f_release["blocking_reasons"]
        )
    }
)

display(remaining_blockers_df)

assert step11f_summary["critical_chain_count"] == 30
assert step11f_summary["defended_critical_count"] == 30
assert step11f_summary[
    "blocked_before_decision_count"
] == 4
assert step11f_summary[
    "rejected_by_assurance_count"
] == 25
assert step11f_summary["quarantined_count"] == 1
assert step11f_summary["undetected_count"] == 0
assert step11f_summary["harness_error_count"] == 0
assert step11f_summary[
    "unsafe_decision_release_count"
] == 0
assert step11f_summary[
    "critical_compound_chain_defence_rate"
] == 1.0
assert step11f_summary[
    "fail_closed_chain_handling_rate"
] == 1.0
assert step11f_summary[
    "multi_control_detection_rate"
] == 1.0
assert step11f_summary[
    "order_sensitive_chain_consistency_rate"
] == 1.0
assert step11f_summary[
    "cross_layer_integrity_defence_rate"
] == 1.0
assert step11f_summary[
    "trusted_input_integrity_passed"
] is True
assert step11f_summary["overall_passed"] is True
assert step11f_release["stage_gate_status"] == "PASS"
assert (
    step11f_release["production_readiness_status"]
    == "BLOCKED"
)

print("Step 11F compound attack assurance checks: PASS")


## Step 11G: Temporal, Replay, and Concurrency Assurance

This stage verifies that stale evidence, replayed events, sequence rollback, policy-version drift, TOCTOU changes, concurrent conflicts and inconsistent snapshots cannot produce an unsafe releasable decision.

All scenarios use an injected deterministic clock. No wall-clock waits or sleep-based timing assumptions influence the assurance result.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

step11g_report_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "temporal"
    / "step11g_temporal_assurance_report.json"
)

step11g_results_path = (
    PROJECT_ROOT
    / "data"
    / "validation_reports"
    / "temporal"
    / "step11g_temporal_results.csv"
)

step11g_report = json.loads(
    step11g_report_path.read_text(encoding="utf-8")
)

step11g_results_df = pd.read_csv(step11g_results_path)
step11g_summary = step11g_report["summary"]
step11g_clock = step11g_report["deterministic_clock"]
step11g_release = step11g_report["release_decision"]

step11g_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Critical scenarios",
            "Observed": step11g_summary[
                "critical_scenario_count"
            ],
        },
        {
            "Metric": "Defended",
            "Observed": step11g_summary[
                "defended_critical_count"
            ],
        },
        {
            "Metric": "Accepted current",
            "Observed": step11g_summary[
                "accepted_current_count"
            ],
        },
        {
            "Metric": "Rejected stale",
            "Observed": step11g_summary[
                "rejected_stale_count"
            ],
        },
        {
            "Metric": "Rejected replay",
            "Observed": step11g_summary[
                "rejected_replay_count"
            ],
        },
        {
            "Metric": "Quarantined conflict",
            "Observed": step11g_summary[
                "quarantined_conflict_count"
            ],
        },
        {
            "Metric": "Blocked version drift",
            "Observed": step11g_summary[
                "blocked_version_drift_count"
            ],
        },
        {
            "Metric": "Blocked integrity change",
            "Observed": step11g_summary[
                "blocked_integrity_change_count"
            ],
        },
        {
            "Metric": "Retry required",
            "Observed": step11g_summary[
                "retry_required_count"
            ],
        },
        {
            "Metric": "Undetected",
            "Observed": step11g_summary[
                "undetected_count"
            ],
        },
        {
            "Metric": "Harness errors",
            "Observed": step11g_summary[
                "harness_error_count"
            ],
        },
        {
            "Metric": "Unsafe decision releases",
            "Observed": step11g_summary[
                "unsafe_decision_release_count"
            ],
        },
    ]
)

display(step11g_summary_df)

display(
    pd.DataFrame(
        [
            {
                "Deterministic evaluation time": (
                    step11g_clock["evaluation_time"]
                ),
                "Wall-clock scenario dependency": (
                    step11g_clock[
                        "wall_clock_reads_used_for_scenarios"
                    ]
                ),
            }
        ]
    )
)


In [ ]:
preferred_columns = [
    "scenario_id",
    "family",
    "handler",
    "severity",
    "outcome",
    "first_blocking_control",
    "unsafe_decision_released",
]

available_columns = [
    column
    for column in preferred_columns
    if column in step11g_results_df.columns
]

display(step11g_results_df[available_columns])

outcome_counts_df = (
    step11g_results_df["outcome"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Count")
)

display(outcome_counts_df)


In [ ]:
temporal_rates_df = pd.DataFrame(
    [
        {
            "Control": "Critical temporal defence",
            "Rate": step11g_summary[
                "critical_temporal_scenario_defence_rate"
            ],
        },
        {
            "Control": "Stale-evidence rejection",
            "Rate": step11g_summary[
                "stale_evidence_rejection_rate"
            ],
        },
        {
            "Control": "Replay detection",
            "Rate": step11g_summary[
                "replay_detection_rate"
            ],
        },
        {
            "Control": "TOCTOU detection",
            "Rate": step11g_summary[
                "toctou_detection_rate"
            ],
        },
        {
            "Control": "Policy-version consistency",
            "Rate": step11g_summary[
                "policy_version_consistency_rate"
            ],
        },
        {
            "Control": "Conflict containment",
            "Rate": step11g_summary[
                "concurrent_conflict_containment_rate"
            ],
        },
        {
            "Control": "Idempotency consistency",
            "Rate": step11g_summary[
                "idempotency_consistency_rate"
            ],
        },
        {
            "Control": "Event-order integrity",
            "Rate": step11g_summary[
                "event_order_integrity_rate"
            ],
        },
    ]
)

display(temporal_rates_df)

remaining_blockers_df = pd.DataFrame(
    {
        "Remaining production blocker": (
            step11g_release["blocking_reasons"]
        )
    }
)

display(remaining_blockers_df)


In [ ]:
assert step11g_summary["critical_scenario_count"] == 36
assert step11g_summary["defended_critical_count"] == 36
assert step11g_summary["accepted_current_count"] == 7
assert step11g_summary["rejected_stale_count"] == 7
assert step11g_summary["rejected_replay_count"] == 5
assert step11g_summary["quarantined_conflict_count"] == 6
assert step11g_summary["blocked_version_drift_count"] == 4
assert step11g_summary["blocked_integrity_change_count"] == 4
assert step11g_summary["retry_required_count"] == 3
assert step11g_summary["undetected_count"] == 0
assert step11g_summary["harness_error_count"] == 0
assert step11g_summary["unsafe_decision_release_count"] == 0

assert step11g_summary[
    "critical_temporal_scenario_defence_rate"
] == 1.0
assert step11g_summary[
    "stale_evidence_rejection_rate"
] == 1.0
assert step11g_summary["replay_detection_rate"] == 1.0
assert step11g_summary["toctou_detection_rate"] == 1.0
assert step11g_summary[
    "policy_version_consistency_rate"
] == 1.0
assert step11g_summary[
    "concurrent_conflict_containment_rate"
] == 1.0
assert step11g_summary[
    "idempotency_consistency_rate"
] == 1.0
assert step11g_summary[
    "event_order_integrity_rate"
] == 1.0

assert step11g_summary[
    "trusted_input_integrity_passed"
] is True
assert step11g_summary["overall_passed"] is True
assert step11g_clock[
    "wall_clock_reads_used_for_scenarios"
] is False
assert step11g_release["stage_gate_status"] == "PASS"
assert (
    step11g_release["production_readiness_status"]
    == "BLOCKED"
)

print("Step 11G temporal assurance checks: PASS")
